In [2]:
import pandas as pd
import numpy as np
import re
import unicodedata

# Load your TSV files
source1 = pd.read_csv("train_source1.tsv", sep="\t")
source2 = pd.read_csv("train_source2.tsv", sep="\t")
source3 = pd.read_csv("train_source3.tsv", sep="\t")

ground_truth = pd.read_csv("train_ground_truth.tsv", sep="\t")

print("Source 1:", source1.shape)
print("Source 2:", source2.shape)
print("Source 3:", source3.shape)
print("Ground truth:", ground_truth.shape)

print("\nColumns:")
print(source1.columns)
print(ground_truth.columns)

Source 1: (2206821, 4)
Source 2: (5034616, 4)
Source 3: (5285603, 4)
Ground truth: (2206821, 2)

Columns:
Index(['entity_id', 'business_name', 'business_address', 'country'], dtype='object')
Index(['source1_entity_id', 'matched_entity_ids'], dtype='object')


In [6]:
display(source1.head())
display(source2.head())
display(source3.head())
display(ground_truth.head())

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [7]:
print("Source 1 missing values:")
print(source1.isnull().sum())

print("\nSource 2 missing values:")
print(source2.isnull().sum())

print("\nSource 3 missing values:")
print(source3.isnull().sum())

print("\nGround truth missing values:")
print(ground_truth.isnull().sum())

Source 1 missing values:
entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

Source 2 missing values:
entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64

Source 3 missing values:
entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64

Ground truth missing values:
source1_entity_id          0
matched_entity_ids    123247
dtype: int64


# Step 1: Normalize efficiently

In [8]:
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFKD", text)

    # Lowercase
    text = text.lower()

    # Remove accents
    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )

    # Replace punctuation with spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [11]:
for df in [source1, source2, source3]:

    df["name_normalized"] = df["business_name"].apply(normalize_text)

    df["address_normalized"] = df["business_address"].apply(normalize_text)

    df["country_normalized"] = df["country"].apply(normalize_text)

In [32]:
source2_small = source2[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
].copy()

source3_small = source3[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
].copy()

source2_small["source"] = "source2"
source3_small["source"] = "source3"

reference = pd.concat(
    [source2_small, source3_small],
    ignore_index=True
)

print(reference.shape)
display(source2_small)
display(source3_small)
display(reference.head(10320219))

(10320219, 5)


,entity_id,business_name,business_address,country,source
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India,source2
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,source2
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India,source2
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US,source2
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US,source2
...,...,...,...,...,...
5034611,S2-147379319,CHOICE SECURITY PARTNERS CO,"IL, 023 Franklin St, DANVILLE",US,source2
5034612,S2-828699622,TMQ Public (Center),"65 Fifth Street, FULTON, NY",US,source2
5034613,S2-726846711,CONTINENTAL CDT PC,"SPENCER, MA, 18 GROVE ST",US,source2
5034614,S2-115058866,लक्ष्मी मीडिया प्राइवेट लिमिटेड,"154, BHUGAON, PUNE, महाराष्ट्र",India,source2


,entity_id,business_name,business_address,country,source
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US,source3
1,S3-859268022,International South Consultants Private Ltd,NaN,India,source3
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US,source3
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US,source3
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India,source3
...,...,...,...,...,...
5285598,S3-549653960,Vip Solutions,NaN,India,source3
5285599,S3-45212707,Cornerstone Nippon,"Paula Drive, Arkansas, N Little Rock",US,source3
5285600,S3-257390113,Legacy Piedmont,"Portland, Maine, Forest Park",US,source3
5285601,S3-97881155,Pranava Meribl Trust,"235, 13Th Cross Road, Hoyasala Ngr 2D Stage, B...",India,source3


,entity_id,business_name,business_address,country,source
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India,source2
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,source2
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India,source2
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US,source2
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US,source2
...,...,...,...,...,...
10320214,S3-549653960,Vip Solutions,NaN,India,source3
10320215,S3-45212707,Cornerstone Nippon,"Paula Drive, Arkansas, N Little Rock",US,source3
10320216,S3-257390113,Legacy Piedmont,"Portland, Maine, Forest Park",US,source3
10320217,S3-97881155,Pranava Meribl Trust,"235, 13Th Cross Road, Hoyasala Ngr 2D Stage, B...",India,source3


# Step 3: Create an exact-match key

In [12]:
source1["exact_key"] = (
    source1["business_name"] + "|" +
    source1["business_address"] + "|" +
    source1["country"]
)

reference["exact_key"] = (
    reference["business_name"] + "|" +
    reference["business_address"] + "|" +
    reference["country"]
)
print(reference.columns)
display(reference.head(10320219))

Index(['entity_id', 'business_name', 'business_address', 'country', 'source',
       'exact_key'],
      dtype='object')


,entity_id,business_name,business_address,country,source,exact_key
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India,source2,राम मार्केटिंग प्राइवेट लिमिटेड|KH NO. -570/13...
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,source2,"-- Holloway Peak Inc Seafood|105 ELM ST, MORGA..."
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India,source2,"आदित्य प्रॉपर्टीज एलएलपी|G-3/571, GULMOHAR COL..."
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US,source2,"Summit Inc|GREENSBORO, NC, 19 1/2 STARDUST TRA..."
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US,source2,"Delta Tetlecommunication Inc|914 PIERPONT AVE,..."
...,...,...,...,...,...,...
10320214,S3-549653960,Vip Solutions,NaN,India,source3,NaN
10320215,S3-45212707,Cornerstone Nippon,"Paula Drive, Arkansas, N Little Rock",US,source3,"Cornerstone Nippon|Paula Drive, Arkansas, N Li..."
10320216,S3-257390113,Legacy Piedmont,"Portland, Maine, Forest Park",US,source3,"Legacy Piedmont|Portland, Maine, Forest Park|US"
10320217,S3-97881155,Pranava Meribl Trust,"235, 13Th Cross Road, Hoyasala Ngr 2D Stage, B...",India,source3,"Pranava Meribl Trust|235, 13Th Cross Road, Hoy..."


# Step 4: Remove useless duplicate keys

In [16]:
reference_unique = reference.drop_duplicates(
    subset=["exact_key"],
    keep=False
)

print("Reference rows:", len(reference))
print("Unique exact keys:", len(reference_unique))
display(reference_unique.head(9887955))

Reference rows: 10320219
Unique exact keys: 9887955


,entity_id,business_name,business_address,country,source,exact_key
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India,source2,राम मार्केटिंग प्राइवेट लिमिटेड|KH NO. -570/13...
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,source2,"-- Holloway Peak Inc Seafood|105 ELM ST, MORGA..."
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India,source2,"आदित्य प्रॉपर्टीज एलएलपी|G-3/571, GULMOHAR COL..."
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US,source2,"Summit Inc|GREENSBORO, NC, 19 1/2 STARDUST TRA..."
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US,source2,"Delta Tetlecommunication Inc|914 PIERPONT AVE,..."
...,...,...,...,...,...,...
10320212,S3-268760762,Interstate Corporation,"00301 Park Ave, Renton, Washington",US,source3,"Interstate Corporation|00301 Park Ave, Renton,..."
10320213,S3-355823625,Solgild Labs D.B.A. GH Certified Dental LLC,"20019 Spring Hill Lane, Orange County, VA",US,source3,Solgild Labs D.B.A. GH Certified Dental LLC|20...
10320215,S3-45212707,Cornerstone Nippon,"Paula Drive, Arkansas, N Little Rock",US,source3,"Cornerstone Nippon|Paula Drive, Arkansas, N Li..."
10320216,S3-257390113,Legacy Piedmont,"Portland, Maine, Forest Park",US,source3,"Legacy Piedmont|Portland, Maine, Forest Park|US"


# Step 5: Exact matching using Pandas merge

In [33]:
exact_matches = source1[
    ["entity_id", "exact_key"]
].merge(
    reference_unique[
        ["exact_key", "entity_id", "source"]
    ],
    on="exact_key",
    how="left",
    suffixes=("_source1", "_matched")
)
print(exact_matches.shape)

exact_matches[exact_matches["entity_id_matched"].notna()]
exact_matches[exact_matches["entity_id_matched"].isna()]

# print(
#     "Unmatched:",
#     exact_matches["entity_id_matched"].isna().sum()
# )
# display(exact_matches)

(2206821, 4)


,entity_id_source1,exact_key,entity_id_matched,source
0,S1-925783039,"Orelee's Barbershop|1795 Westchester Drive, Hi...",NaN,NaN
1,S1-773889195,"Prime Money|17560 Ellis Road, Tahlequah, OK|US",NaN,NaN
2,S1-377745466,"B+ Retail Inc|1712 Montebello Avenue, Phoenix,...",NaN,NaN
3,S1-133037285,"Christ Chapel|2100 Cameron Drive, Unit APARTME...",NaN,NaN
4,S1-755362802,"Prabhav Business Center|797, Lake Town Block A...",NaN,NaN
...,...,...,...,...
2206816,S1-479751630,Arlyn Farooqi Prairie Medical|16804 Stevenage ...,NaN,NaN
2206817,S1-392755726,"Patriot Fund|MD, 7709 Ashdale Road, Capitol He...",NaN,NaN
2206818,S1-716147977,"Sloan Fact of Yonkers|28 Armstrong Avenue, Uni...",NaN,NaN
2206819,S1-770136229,Ritter's Machine Works|5609 Spring Meadow Road...,NaN,NaN


In [28]:
ground_truth["matched_entity_ids"] = (
    ground_truth["matched_entity_ids"]
    .fillna("")
    .astype(str)
)
gt = ground_truth.set_index(
    "source1_entity_id"
)["matched_entity_ids"]
exact_matches["ground_truth"] = (
    exact_matches["entity_id_source1"].map(gt)
)
display(
    exact_matches[
        [
            "entity_id_source1",
            "entity_id_matched",
            "ground_truth"
        ]
    ].head(20)
)

,entity_id_source1,entity_id_matched,ground_truth
0,S1-925783039,NaN,"S2-157377754,S2-517291332,S3-997698194,S3-6981..."
1,S1-773889195,NaN,"S2-970528089,S2-374107005,S3-189140669,S3-1614..."
2,S1-377745466,NaN,"S2-203037501,S3-70942743,S3-402918963,S3-83617..."
3,S1-133037285,NaN,"S2-407207105,S2-103437512,S3-476003339,S3-3314..."
4,S1-755362802,NaN,"S2-587230276,S3-440254853"
5,S1-851869949,NaN,"S2-915265864,S2-801813753,S2-961342067,S3-9498..."
6,S1-785847572,NaN,"S2-682432196,S3-943437724,S3-145991649,S3-9481..."
7,S1-27541239,NaN,"S2-199341916,S2-85756610,S3-893131105,S3-73975..."
8,S1-629417405,NaN,"S2-522538408,S2-928426462,S3-613837325,S3-6069..."
9,S1-22305073,NaN,"S2-578011604,S3-221070872,S3-183589405,S3-3170..."
